In [20]:
import torch
import torch.nn as nn

# Sample Bidirectional RNN Model
rnn = nn.RNN(input_size=5, hidden_size=2, num_layers=1, bidirectional=True)

In [21]:
# Paremeters Order
forward_processing_wts, reverse_processing_wts = rnn.all_weights

current_ts_wts, hidden_ts_wts, current_ts_bias, hidden_ts_bias = forward_processing_wts
rev_current_ts_wts, rev_hidden_ts_wts, rev_current_ts_bias, rev_hidden_ts_bias = reverse_processing_wts

print(forward_processing_wts, reverse_processing_wts, sep='\n\n')

[Parameter containing:
tensor([[-0.5711, -0.4711,  0.0905,  0.1210,  0.1176],
        [-0.1336,  0.5534,  0.2329, -0.4281,  0.4653]], requires_grad=True), Parameter containing:
tensor([[ 0.1949, -0.5776],
        [ 0.5335,  0.4518]], requires_grad=True), Parameter containing:
tensor([ 0.2654, -0.2180], requires_grad=True), Parameter containing:
tensor([ 0.0120, -0.0724], requires_grad=True)]

[Parameter containing:
tensor([[-0.2513,  0.2362, -0.0594, -0.5293,  0.6794],
        [ 0.2532, -0.1292, -0.2708, -0.5333,  0.3655]], requires_grad=True), Parameter containing:
tensor([[-0.0907,  0.5199],
        [ 0.5507,  0.0479]], requires_grad=True), Parameter containing:
tensor([0.6467, 0.1320], requires_grad=True), Parameter containing:
tensor([-0.1806, -0.1647], requires_grad=True)]


### Bidirectional RNNs: How Backpropagation Works (Theoretical Deep Dive)

#### The Short Answer

**Two completely separate RNN layers** — one forward, one backward — each with **their own independent weights** — are trained simultaneously using a modified version of Backpropagation Through Time (BPTT). They do NOT share weights. They do NOT interact during forward or backward passes. They are two distinct networks whose outputs are only combined at the final output layer .

---

#### The Architecture: Two Independent Networks

Imagine you have two employees:

- **Employee A (Forward RNN)**: Reads a document from page 1 to the end, taking notes as they go
- **Employee B (Backward RNN)**: Reads the SAME document from the last page back to page 1, taking notes as they go

They sit in **separate cubicles**, use **separate notebooks**, and **never talk to each other** during reading. Only at the very end do they combine their notes for the final report .

This is exactly how a Bi-RNN works:

| Component | Forward RNN | Backward RNN |
|---|---|---|
| **Direction** | Left → Right | Right → Left |
| **Hidden states** | $\vec{h}_1, \vec{h}_2, ..., \vec{h}_T$ | $\overleftarrow{h}_1, \overleftarrow{h}_2, ..., \overleftarrow{h}_T$ |
| **Input weights** | $\mathbf{W}_{xh}^{(f)}$ | $\mathbf{W}_{xh}^{(b)}$ |
| **Recurrent weights** | $\mathbf{W}_{hh}^{(f)}$ | $\mathbf{W}_{hh}^{(b)}$ |
| **Biases** | $\mathbf{b}_h^{(f)}$ | $\mathbf{b}_h^{(b)}$ |
| **Interaction with other direction** | **NONE** | **NONE** |

From the Dive into Deep Learning textbook: *"The forward and backward hidden state updates... where the weights $\mathbf{W}_{xh}^{(f)}, \mathbf{W}_{hh}^{(f)}, \mathbf{W}_{xh}^{(b)}, \mathbf{W}_{hh}^{(b)}$ and biases $\mathbf{b}_h^{(f)}, \mathbf{b}_h^{(b)}$ are all the model parameters"* — notice the **separate superscripts (f) and (b)** indicating independent parameters .

---

#### Why They Must Be Separate

##### 1. **Different Temporal Dependencies**

The forward RNN learns: *"Given what I've seen so far, what comes next?"*
The backward RNN learns: *"Given what comes after, what should this be?"*

These are fundamentally different questions requiring different weights. If they shared weights, the network would be forced to use the same logic for both past-to-future and future-to-past reasoning — which makes no sense .

##### 2. **Different Information Flow**

```
FORWARD RNN:                    BACKWARD RNN:
h→1 ← h→2 ← h→3 ← ... ← h→T    h←T ← h←T-1 ← ... ← h←2 ← h←1
   (past informs future)           (future informs past)
```

The forward RNN's hidden state at time $t$ depends on $t-1$. The backward RNN's hidden state at time $t$ depends on $t+1$. You cannot use the same recurrent weights for both directions because the **direction of information flow is opposite** .

---

#### How Backpropagation Works: The Full Picture

##### Step 1: Forward Pass (Two Independent Passes)

During training, for a sequence of length $T$:

**Forward RNN** computes left-to-right:
- $\vec{h}_1 = \text{RNN}(x_1, \vec{h}_0)$
- $\vec{h}_2 = \text{RNN}(x_2, \vec{h}_1)$
- ...
- $\vec{h}_T = \text{RNN}(x_T, \vec{h}_{T-1})$

**Backward RNN** computes right-to-left:
- $\overleftarrow{h}_T = \text{RNN}(x_T, \overleftarrow{h}_{T+1})$
- $\overleftarrow{h}_{T-1} = \text{RNN}(x_{T-1}, \overleftarrow{h}_T)$
- ...
- $\overleftarrow{h}_1 = \text{RNN}(x_1, \overleftarrow{h}_2)$

**At each time step $t$**, the two hidden states are concatenated:
$$h_t = [\vec{h}_t ; \overleftarrow{h}_t]$$

This concatenated vector goes to the output layer to produce the prediction .

##### Step 2: Loss Computation

The loss is computed by comparing predictions to ground truth. This single loss value is the **starting point** for backpropagation.

##### Step 3: Backward Pass — The Critical Part

Here's where it gets interesting. The loss gradient flows back to **both** RNNs, but **through separate, independent paths**:

```
Output Layer
    │
    ├──► Gradient flows to Forward RNN output weights
    │       │
    │       ▼
    │   Forward Hidden States [h→1, h→2, ..., h→T]
    │       │
    │       ▼
    │   Backpropagate Through Time (LEFT-TO-RIGHT in reverse)
    │   Update: W_xh^(f), W_hh^(f), b_h^(f)
    │
    └──► Gradient flows to Backward RNN output weights
            │
            ▼
        Backward Hidden States [h←1, h←2, ..., h←T]
            │
            ▼
        Backpropagate Through Time (RIGHT-TO-LEFT in reverse)
        Update: W_xh^(b), W_hh^(b), b_h^(b)
```

**Key insight from GeeksforGeeks**: *"In a Bidirectional RNN however, since there are forward and backward passes happening simultaneously, updating the weights for the two processes could happen at the same point in time. This leads to erroneous results. Thus, to accommodate forward and backward passes separately"* — the algorithm processes forward states from $t=N$ to $1$ and backward states from $t=1$ to $N$ during backpropagation .

##### Step 4: Two Separate BPTT Executions

**For the Forward RNN:**
- Start from the loss gradient at the output
- Backpropagate through the output layer weights → get gradient w.r.t. $\vec{h}_t$
- Apply standard BPTT: propagate gradients from $t=T$ back to $t=1$
- Update $\mathbf{W}_{xh}^{(f)}$, $\mathbf{W}_{hh}^{(f)}$, $\mathbf{b}_h^{(f)}$

**For the Backward RNN:**
- Start from the SAME loss gradient at the output
- Backpropagate through the output layer weights → get gradient w.r.t. $\overleftarrow{h}_t$
- Apply standard BPTT: propagate gradients from $t=1$ back to $t=T$ (opposite direction!)
- Update $\mathbf{W}_{xh}^{(b)}$, $\mathbf{W}_{hh}^{(b)}$, $\mathbf{b}_h^{(b)}$

From the academic paper on He-BiLSTM: *"In traditional BiLSTM models, there are 24 parameters that need to be learned through backpropagation. Specifically, there are 12 parameters for the positive direction... and there is another set of 12 parameters for the negative direction"* — confirming completely separate parameter sets .

---

#### The "Unrolling" Intuition

Think of BPTT as "unrolling" the RNN across time into one giant feedforward network:

**Unrolled Forward RNN:**
```
x1 → [RNN cell] → x2 → [RNN cell] → x3 → ... → xT
        ↑              ↑              ↑
       h→1            h→2            h→3
```
(Unrolled into T layers, all sharing the same weights)

**Unrolled Backward RNN:**
```
xT → [RNN cell] → xT-1 → [RNN cell] → xT-2 → ... → x1
        ↑                ↑                ↑
       h←T             h←T-1           h←T-2
```
(Also unrolled into T layers, but with DIFFERENT weights)

When you unroll both and connect them to the same output layer, you get one big feedforward network. Standard backpropagation works on this unrolled graph — but the gradients for the forward path and backward path never mix because there are **no connections between them** except at the output layer .

---

#### Boundary Conditions: The Special Cases

From the original Schuster & Paliwal BRNN paper: *"The forward state inputs at t=1 and the backward state inputs at t=T are not known. Setting these could be made part of the learning process, but here, they are set arbitrarily to a fixed value (0.5). In addition, the local state derivatives at t=T for the forward states and at t=1 for the backward states are not known and are set here to zero"* .

This means:
- **Forward RNN**: Initial hidden state $\vec{h}_0$ is typically zero or learned; gradient at $t=T$ is zero (no future state to propagate to)
- **Backward RNN**: Initial hidden state $\overleftarrow{h}_{T+1}$ is typically zero or learned; gradient at $t=1$ is zero (no past state to propagate to)

---

#### Summary: Why This Design Works

| Question | Answer |
|---|---|
| **One layer or two?** | **Two separate layers** — forward and backward are completely independent RNNs |
| **Shared weights?** | **No** — each direction has its own $\mathbf{W}_{xh}$, $\mathbf{W}_{hh}$, and $\mathbf{b}$ |
| **How does backprop work?** | **Two separate BPTT passes** — one for each direction, gradients computed independently |
| **Do gradients mix between directions?** | **No** — gradients only meet at the output layer, then flow back through separate paths |
| **Why not one layer doing both?** | **Impossible** — an RNN cell can only pass state in one direction; you need two cells for two directions |

The bidirectional RNN is not a single RNN that magically reads both ways. It is **two RNNs running in parallel**, each learning different aspects of the sequence, whose outputs are concatenated to give each time step a complete view of its context .